In [ ]:
!pip install minatar

# 05_final_comparison.ipynb

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import imageio

from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/rl-final-project"
print("Drive:", DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/rl-final-project


In [ ]:
import minatar
import numpy as np

class MiniAtariWrapper:
    def __init__(self):
        self.env = minatar.Environment("breakout")
        self.n_actions = self.env.num_actions()
        self.state_shape = (100,)
    def reset(self):
        self.env.reset()
        return self._preprocess(self.env.state())
    def step(self, action):
        r, done = self.env.act(int(action))
        return self._preprocess(self.env.state()), float(r), bool(done)
    def render_raw(self):
        st = self.env.state()
        arr = np.array(st, dtype=np.float32)
        if arr.ndim == 3:
            arr = arr.sum(axis=2)
        mn, mx = arr.min(), arr.max()
        rng = mx - mn if mx > mn else 1
        arr = ((arr - mn)/rng*255).astype(np.uint8)
        return arr
    def _preprocess(self, st):
        arr = np.array(st, dtype=np.float32)
        if arr.ndim == 3:
            arr = arr.sum(axis=2)
        mn, mx = arr.min(), arr.max()
        rng = mx - mn if mx > mn else 1
        arr = (arr - mn)/rng
        return arr.flatten()

In [ ]:
def eval_policy(env, policy_fn, episodes=20, steps=400):
    scores = []
    for _ in range(episodes):
        s = env.reset()
        total = 0.0
        for t in range(steps):
            a = policy_fn(s)
            s, r, done = env.step(a)
            total += r
            if done:
                break
        scores.append(total)
    return np.array(scores)

In [ ]:
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MLP(nn.Module):
    def __init__(self, inp, hidden=[256,128], out=None):
        super().__init__()
        layers = []
        last = inp
        for h in hidden:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        if out is not None:
            layers.append(nn.Linear(last, out))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class Actor(nn.Module):
    def __init__(self, inp, hidden=[256,128], actions=6):
        super().__init__()
        self.pi = MLP(inp, hidden, actions)
    def forward(self, x):
        logits = self.pi(x)
        return torch.softmax(logits, dim=-1)

In [ ]:
# --- FIX A2C STATE DICT (remove 'mlp.' prefix) ---

raw_sd = torch.load(a2c_actor_path, map_location=device)

fixed_sd = {}
for k, v in raw_sd.items():
    new_key = k.replace("mlp.", "")   # <- remove prefix
    fixed_sd[new_key] = v

# save temporary fixed version
fixed_a2c_path = f"{DRIVE}/policy_gradient/a2c_actor_fixed.pth"
torch.save(fixed_sd, fixed_a2c_path)

print("A2C keys fixed and saved to:", fixed_a2c_path)

A2C keys fixed and saved to: /content/drive/MyDrive/rl-final-project/policy_gradient/a2c_actor_fixed.pth


In [ ]:
# --- Paths ---
dqn_path = f"{DRIVE}/dqn/best_model.pth"
bc_path = f"{DRIVE}/imitation/bc_model.pth"
bcq_q_path = f"{DRIVE}/offline_rl/bcq_q_model.pth"
a2c_actor_path = f"{DRIVE}/policy_gradient/a2c_actor_fixed.pth"
ppo_policy_path = f"{DRIVE}/policy_gradient/ppo_policy.pth"
shaped_path = f"{DRIVE}/shaped_dqn/best_model.pth" if os.path.exists(f"{DRIVE}/shaped_dqn/best_model.pth") else None

env = MiniAtariWrapper()

# --- Load DQN ---
dqn = MLP(100, [256,128], out=env.n_actions).to(device)
dqn.load_state_dict(torch.load(dqn_path, map_location=device))
dqn.eval()

# --- Load BC ---
bc = MLP(100, [256,128], out=env.n_actions).to(device)
bc.load_state_dict(torch.load(bc_path, map_location=device))
bc.eval()

# --- Load BCQ behavior model ---
bcq = MLP(100, [256,128], out=env.n_actions).to(device)
bcq.load_state_dict(torch.load(bcq_q_path, map_location=device))
bcq.eval()

# --- Load A2C ---
a2c = MLP(100, [256,128], out=env.n_actions).to(device)
a2c.load_state_dict(torch.load(a2c_actor_path, map_location=device))
a2c.eval()


# --- Load PPO ---
ppo = MLP(100, [256,128], out=env.n_actions).to(device)
ppo.load_state_dict(torch.load(ppo_policy_path, map_location=device))
ppo.eval()

# --- Load Shaped DQN (if exists) ---
if shaped_path:
    shaped = MLP(100, [256,128], out=env.n_actions).to(device)
    shaped.load_state_dict(torch.load(shaped_path, map_location=device))
    shaped.eval()
else:
    shaped = None

print("All models loaded.")

All models loaded.


In [ ]:
def greedy(model):
    def fn(s):
        with torch.no_grad():
            x = torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)
            out = model(x)
        return int(torch.argmax(out).item())
    return fn

policies = {
    "DQN": greedy(dqn),
    "BC": greedy(bc),
    "BCQ": greedy(bcq),
    "A2C": lambda s: int(torch.argmax(a2c(torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device))).item()),
    "PPO": greedy(ppo)
}

if shaped is not None:
    policies["Shaped DQN"] = greedy(shaped)

print("Policies ready:", list(policies.keys()))

Policies ready: ['DQN', 'BC', 'BCQ', 'A2C', 'PPO', 'Shaped DQN']


In [ ]:
env_eval = MiniAtariWrapper()

scores = {}
for name, pol in policies.items():
    print(f"Evaluating {name}...")
    sc = eval_policy(env_eval, pol, episodes=20)
    scores[name] = sc
    print(name, "mean:", np.mean(sc), "std:", np.std(sc))

Evaluating DQN...
DQN mean: 0.4 std: 0.48989794855663565
Evaluating BC...
BC mean: 1.55 std: 1.116915395184434
Evaluating BCQ...
BCQ mean: 0.65 std: 0.47696960070847283
Evaluating A2C...
A2C mean: 3.0 std: 0.9486832980505138
Evaluating PPO...
PPO mean: 5.3 std: 1.307669683062202
Evaluating Shaped DQN...
Shaped DQN mean: 0.05 std: 0.21794494717703364


In [ ]:
labels = list(scores.keys())
data = [scores[k] for k in labels]

plt.figure(figsize=(10,6))
plt.boxplot(data, labels=labels)
plt.title("Final Comparison of All RL Agents (20 episodes)")
plt.ylabel("Episode Return")
plt.grid(True)

out = f"{DRIVE}/plots/comparison/final_boxplot.png"
plt.savefig(out, dpi=150)
plt.close()

print("Saved:", out)

/tmp/ipython-input-45454531.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels)


Saved: /content/drive/MyDrive/rl-final-project/plots/comparison/final_boxplot.png


In [ ]:
means = [np.mean(scores[k]) for k in labels]
stds = [np.std(scores[k]) for k in labels]

plt.figure(figsize=(10,6))
plt.bar(labels, means, yerr=stds, capsize=6)
plt.title("Mean Reward ± Std — All Agents")
plt.ylabel("Mean Return")

out = f"{DRIVE}/plots/comparison/final_bar.png"
plt.savefig(out, dpi=150)
plt.close()

print("Saved:", out)

Saved: /content/drive/MyDrive/rl-final-project/plots/comparison/final_bar.png


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Agent": labels,
    "Mean Reward": means,
    "Std Reward": stds
})

csv_path = f"{DRIVE}/plots/comparison/final_scores.csv"
df.to_csv(csv_path, index=False)

df

,Agent,Mean Reward,Std Reward
0,DQN,0.40,0.489898
1,BC,1.55,1.116915
2,BCQ,0.65,0.476970
3,A2C,3.00,0.948683
4,PPO,5.30,1.307670
5,Shaped DQN,0.05,0.217945


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def radar_plot(labels, values, title, save_path):
    N = len(labels)

    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    values_cycle = np.concatenate((values, [values[0]]))
    angles_cycle = angles + [angles[0]]

    plt.figure(figsize=(8,8))
    ax = plt.subplot(111, polar=True)

    ax.plot(angles_cycle, values_cycle, linewidth=2, linestyle='solid')
    ax.fill(angles_cycle, values_cycle, alpha=0.25)

    ax.set_thetagrids(np.degrees(angles), labels)
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)

    plt.savefig(save_path, dpi=150)
    plt.close()
    print("Saved radar chart:", save_path)

# --- Generate radar chart ---
radar_path = f"{DRIVE}/plots/comparison/final_radar.png"
radar_plot(labels, means, "Final Agent Comparison — Radar Chart", radar_path)

Saved radar chart: /content/drive/MyDrive/rl-final-project/plots/comparison/final_radar.png
